In [223]:
import pandas as pd
import os
import warnings
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')


def load_dfs_from_parquet(folder_path):
  """
  Loads all parquet files within a folder into a dictionary of DataFrames.

  Args:
    folder_path: The path to the folder containing parquet files.

  Returns:
    A dictionary where keys are parquet file names (without extension) and
    values are corresponding pandas DataFrames.
  """
  dfs = {}
  for filename in os.listdir(folder_path):
    if filename.endswith(".parquet"):
      file_path = os.path.join(folder_path, filename)
      df_name = filename[:-8]  # Remove ".parquet" from filename
      dfs[df_name] = pd.read_parquet(file_path)
  return dfs


folder_path = "data/processed"
dfs_dict = load_dfs_from_parquet(folder_path)

orders_df= dfs_dict['cleaned_orders_table']
payments = dfs_dict['cleaned_order_payments_table']
reviews = dfs_dict['cleaned_reviews_table']
products = dfs_dict['cleaned_product_table']
customers = dfs_dict['cleaned_customer_table']
customers.drop('customer_zip_code_prefix', axis=1, inplace=True, errors='ignore')
order_items = dfs_dict['cleaned_order_items_table']

orders = orders_df.merge(customers, on='customer_id', how='left').merge(payments, on='order_id', how='left')
order_items_products = order_items.merge(products, on='product_id', how='left')

1️⃣ Customer Behavior & Sales Trends

💡 Goal: Understand who the customers are and what they buy.

✅ Business Questions:

1. Which cities and states generate the most sales?

2. What is the average order value (AOV) per customer?

3. Do customers tend to buy high-value products or multiple low-value products?

4. What is the most common payment method used by customers?

5. How do installment plans affect the total order value? (e.g., do higher installment plans lead to more expensive purchases?)

6. Are there any seasonal trends in sales? (e.g., spikes during holidays?)

In [224]:
import plotly.express as px
city_sales = orders.groupby(['customer_state'])['order_id'].count().reset_index(name='total_orders')
city_sales = city_sales.sort_values(by='total_orders', ascending=True)  # Ascending so the largest is at the top in a horizontal bar chart
fig_city_sales = px.bar(city_sales, y='customer_state', x='total_orders', title='Sales by State', orientation='h', text_auto=True)

# Improve bar visibility & layout
fig_city_sales.update_layout(
    yaxis=dict(categoryorder='total ascending'),  # Ensure labels are in order
    height=600,  # Increase figure height so all labels fit
    margin=dict(l=150),  # Add margin for long labels
)

# Make bars thicker
fig_city_sales.update_traces(
    textposition='outside',
    marker=dict(line=dict(width=2))  # Add slight border for better visibility
)
fig_city_sales.show()

# 2. Average Order Value (AOV)
aov = orders.groupby('customer_id')['payment_value'].mean().reset_index()
fig_aov = px.histogram(aov, x='payment_value', nbins=30, title='Average Order Value Distribution')
fig_aov.show()

# 3. High-value vs. multiple low-value purchases
product_sales = order_items.groupby('order_id').agg({'price': ['sum', 'count']}).reset_index()

# Rename columns to remove multi-index
product_sales.columns = ['order_id', 'total_order_value', 'num_items']

# Create scatter plot with trend line
fig_product_sales = px.scatter(
    product_sales,
    x='num_items',
    y='total_order_value',
    title='High-Value vs Multiple Low-Value Purchases',
    labels={'num_items': 'Number of Items', 'total_order_value': 'Total Order Value'},
    trendline='ols',  # Ordinary Least Squares Regression (linear trend)
    opacity=0.6,  # Reduce opacity for better visibility
)

# Customize layout
fig_product_sales.update_traces(marker=dict(size=7))  # Increase dot size
fig_product_sales.update_layout(
    height=600,
    xaxis=dict(title='Number of Items', tickmode='linear'),
    yaxis=dict(title='Total Order Value')
)

fig_product_sales.show()

# 4. Most common payment method
payment_counts = payments['payment_type'].value_counts().reset_index()
fig_payments = px.pie(payment_counts, names='payment_type', values='count', title='Most Common Payment Methods')
fig_payments.show()

# 5. Installment plans effect on order value

installment_effect = payments.groupby('payment_installments')['payment_value'].mean().reset_index()
fig_installments = px.scatter(
    installment_effect,
    x='payment_installments',
    y='payment_value',
    title='Effect of Installments on Order Value',
    trendline='ols',  # Ordinary Least Squares regression line
    labels={'payment_installments': 'Number of Installments', 'payment_value': 'Average Order Value'}
)

fig_installments.show()


# 6. Seasonal sales trends
delivered_orders = orders[orders['order_status'] == 'delivered']
delivered_orders['order_purchase_timestamp'] = pd.to_datetime(delivered_orders['order_purchase_timestamp'])
delivered_orders['month'] = delivered_orders['order_purchase_timestamp'].dt.to_period('M')
seasonal_sales = delivered_orders.groupby('month')['payment_value'].sum().reset_index()

# Convert 'month' column to string before plotting
seasonal_sales['month'] = seasonal_sales['month'].astype(str)

fig_seasonal = px.line(seasonal_sales, x='month', y='payment_value', title='Seasonal Sales Trends')
fig_seasonal.show()




 Key Findings:

 Top Sales Regions:
- The states with the highest number of orders in the past three years are:
  - São Paulo (SP): 44,000 orders
  - Rio de Janeiro (RJ): 13,000 orders
  - Minas Gerais (MG): 12,000 orders

- The states with the lowest number of orders are:
  - Roraima (RR), Amapá (AP), and Acre (AC), each with fewer than 100 orders.

Average Order Value (AOV):
- The histogram analysis indicates that most AOVs are below $250.

 Purchase Behavior:
- A positive trend exists between the number of items in an order and the total order value, indicating that customers who spend more tend to buy multiple low-value products rather than high-value individual items.

 Payment Methods:
- Credit cards are the most commonly used payment method.

 Installment Plans:
- There is a correlation between the number of installments and the average order value. Higher installment plans are associated with higher order values.

 Seasonal Sales Trends:
- Peak sales occur in November, suggesting a strong Black Friday effect.

2️⃣ Product Performance & Pricing

💡 Goal: Identify the best-selling products and optimize pricing.

✅ Questions:

* What are the top-selling product categories?

* Are there certain products that frequently get returned (low review scores, canceled orders)?

* What is the distribution of product prices?

* How does product price affect freight cost?

* Are expensive products delivered faster or slower than cheap ones?

* Are certain product categories more profitable than others?

In [225]:
# Count number of sales per category
top_categories = order_items_products.groupby('product_category_name_english')['order_id'].count().reset_index(name='total_orders')

top_categories = top_categories.sort_values(by='total_orders', ascending=False).head(10)

# Plot bar chart
fig_top_categories = px.bar(
    top_categories,
    x='product_category_name_english',
    y='total_orders',
    title='Top-Selling Product Categories',
    labels={'total_orders': 'Total Orders', 'product_category_name_english': 'Category'},
    text_auto=True
)

fig_top_categories.show()

# Low review score products
products_review_df=order_items_products.merge(reviews, on='order_id', how='left')#[['product_id', 'product_category_name_english', 'review_score']]
low_review_products = products_review_df[products_review_df['review_score'] <= 2].groupby('product_id').size().reset_index(name='low_reviews')

# Canceled orders
orders_product = orders.merge(order_items_products, on='order_id', how='left')
canceled_orders = orders_product[orders_product['order_status'] == 'canceled'].groupby('product_id').size().reset_index(name='canceled_orders')

# Merge with product data
returned_products = low_review_products.merge(canceled_orders, on='product_id', how='outer').fillna(0)
returned_products = returned_products.merge(products[['product_id', 'product_category_name_english']], on='product_id')

# Aggregate by category
top_returns = returned_products.groupby('product_category_name_english')[['low_reviews', 'canceled_orders']].sum().reset_index()
top_returns = top_returns.sort_values(by=['low_reviews', 'canceled_orders'], ascending=False).head(10)

# Plot grouped bar chart
fig_returns = px.bar(
    top_returns,
    x='product_category_name_english',
    y=['low_reviews', 'canceled_orders'],
    barmode='group',
    title='Products with High Returns and Low Reviews',
    labels={'value': 'Count', 'variable': 'Return Type'}
)

fig_returns.show()

fig_price_dist = px.histogram(
    order_items_products,
    x='price',
    nbins=50,
    title='Distribution of Product Prices',
    labels={'price': 'Product Price'},
    marginal='box'
)
fig_price_dist.show()

fig_price_freight = px.scatter(
    order_items,
    x='price',
    y='freight_value',
    title='Product Price vs Freight Cost',
    labels={'price': 'Product Price', 'freight_value': 'Freight Cost'},
    trendline='ols',
    opacity=0.6
)

fig_price_freight.show()

# Create delivery time column
orders_delivered = orders[orders['order_status'] == 'delivered']
orders_delivered['delivery_days'] = (orders_delivered['order_delivered_customer_date'] - orders_delivered['order_purchase_timestamp']).dt.days

# Remove negative and missing values
orders_delivered = orders_delivered[(orders_delivered['delivery_days'] >= 0) & (~orders_delivered['delivery_days'].isna())]

# Merge with order_items
order_data = order_items.merge(orders_delivered[['order_id', 'delivery_days']], on='order_id')

# Scatter plot
fig_price_delivery = px.scatter(
    order_data,
    x='price',
    y='delivery_days',
    title='Does Product Price Affect Delivery Speed?',
    labels={'price': 'Product Price', 'delivery_days': 'Delivery Time (Days)'},
    trendline='ols',
    opacity=0.6
)

fig_price_delivery.show()





 Key Findings:

 Top-Selling Product Categories:
- The best-selling product categories are:
  - Bed-Bath-Table: 11,000 orders
  - Health & Beauty: 10,000 orders
  - Sports & Leisure: 8,000 orders

- The least ordered categories are:
  - Auto, Garden Tools, and Telephony, each with 4,000 orders.

 Returns & Cancellations:
- Housewares have the highest return rate.

Price Distribution:
- Most products are priced at around $100.

4️⃣ Customer Satisfaction & Reviews

💡 Goal: Understand what drives positive and negative customer feedback.

✅ Questions:

* What is the distribution of review scores?

* Do customers give lower ratings when deliveries are late?

* Do different product categories have different average review scores?

* Are certain sellers getting consistently bad reviews?


In [226]:
fig_review_dist = px.histogram(
    reviews,
    x='review_score',
    nbins=5,
    title='Distribution of Customer Review Scores',
    labels={'review_score': 'Review Score'},
    text_auto=True
)

fig_review_dist.show()

#  Merge reviews with orders
review_delivery = reviews.merge(orders_delivered[['order_id', 'delivery_days']], on='order_id')

# Scatter plot
fig_review_delivery = px.scatter(
    review_delivery,
    y='delivery_days',
    x='review_score',
    title='Do Late Deliveries Lead to Lower Ratings?',
    labels={'delivery_days': 'Delivery Time (Days)', 'review_score': 'Review Score'},
    trendline='ols',
    opacity=0.6
)

fig_review_delivery.show()

# Merge reviews with products
category_reviews = reviews.merge(order_items_products, on='order_id')
# category_reviews = category_reviews.merge(products[['product_id', 'product_category_name']], on='product_id')

# Calculate average review score per category
category_avg_reviews = category_reviews.groupby('product_category_name_english')['review_score'].mean().reset_index()
category_avg_reviews['review_score'] = category_avg_reviews['review_score'].round(1)
category_avg_reviews = category_avg_reviews.sort_values(by='review_score', ascending=True)

# Plot bar chart
fig_category_reviews = px.bar(
    category_avg_reviews,
    y='product_category_name_english',
    x='review_score',
    orientation='h',
    title='Average Review Score by Product Category',
    labels={'review_score': 'Average Review Score', 'product_category_name_english': 'Category'},
    text_auto=True
)

# fig_category_reviews.show()

# Improve bar visibility & layout
fig_category_reviews.update_layout(
    yaxis=dict(categoryorder='total ascending'),  # Ensure labels are in order
    height=600,  # Increase figure height so all labels fit
    margin=dict(l=150),  # Add margin for long labels
)

# Make bars thicker
fig_category_reviews.update_traces(
    textposition='outside',
    marker=dict(line=dict(width=2))  # Add slight border for better visibility
)
fig_category_reviews.show()




Key Findings:

 Review Score Distribution:
- Most reviews are rated 4 and 5 stars.

Impact of Late Deliveries on Ratings:
- Late deliveries do not significantly affect customer ratings.

 Review Scores by Category:
- The most rated category is Fashion (Children's Clothes).
- The least rated category, with an average rating of 2.5, is Security & Services.